# 大數據隱私保護、安全與合規

## 學習目標

完成本 Notebook 後，你將能夠：

1. 區分直接識別資訊、準識別資訊與敏感資料。
2. 理解資料再識別風險如何從欄位組合與外部資料比對產生。
3. 使用 Python 實作常見匿名化方法：遮蔽、雜湊、泛化、分桶與擾動。
4. 觀察 k-匿名如何降低個體被唯一識別的風險。
5. 以簡化方式模擬差分隱私中的噪音加入概念。
6. 建立 AI 訓練資料合規檢查的基本思維。


In [ ]:
# ── 環境設定 ────────────────────────────────────
# 載入本章節所需的 Python 套件，並建立一份模擬的顧客資料集，後續將用來示範匿名化與隱私風險評估。

import numpy as np
import pandas as pd
import hashlib
import matplotlib.pyplot as plt

np.random.seed(42)

data = pd.DataFrame({
    "name": ["王小明", "李雅婷", "陳志豪", "林怡君", "張家豪", "黃思涵", "吳宗翰", "周佩蓉"],
    "email": [
        "ming@example.com", "ya@example.com", "hao@example.com", "yi@example.com",
        "chang@example.com", "han@example.com", "wu@example.com", "chou@example.com"
    ],
    "gender": ["男", "女", "男", "女", "男", "女", "男", "女"],
    "age": [29, 31, 32, 45, 47, 46, 63, 64],
    "zipcode": ["100", "100", "104", "220", "220", "221", "800", "800"],
    "purchase_amount": [1200, 1500, 1800, 3200, 2800, 3500, 7600, 8100],
    "disease_risk": ["低", "低", "中", "中", "高", "高", "高", "中"]
})

print("原始資料：")
display(data)


## 核心概念說明

在 AI 與大數據應用中，資料常被分為以下幾類：

- **直接識別資訊**：可直接指向個人的欄位，例如姓名、Email、電話、身分證字號、裝置 ID。
- **準識別資訊**：單獨看未必能識別個人，但組合後可能造成再識別，例如年齡、性別、郵遞區號、瀏覽紀錄、交易時間。
- **敏感資料**：一旦外洩可能造成重大權益損害的資料，例如健康紀錄、信用狀況、宗教信仰、政治立場。

匿名化不是只刪除姓名而已。若保留過於精細的年齡、地區、時間或行為軌跡，攻擊者仍可能透過外部資料比對找出個人。因此，實務上常結合遮蔽、雜湊、泛化、分桶、擾動、k-匿名與差分隱私等方法，在資料可用性與隱私保護之間取得平衡。


In [ ]:
# ── 示範：基礎匿名化技術 ──────────────────────────────
# 這段程式碼示範遮蔽、雜湊、年齡泛化、金額分桶與數值擾動，觀察不同方法如何降低個人資料暴露風險。

import numpy as np
import pandas as pd
import hashlib

np.random.seed(42)

data = pd.DataFrame({
    "name": ["王小明", "李雅婷", "陳志豪", "林怡君", "張家豪", "黃思涵", "吳宗翰", "周佩蓉"],
    "email": [
        "ming@example.com", "ya@example.com", "hao@example.com", "yi@example.com",
        "chang@example.com", "han@example.com", "wu@example.com", "chou@example.com"
    ],
    "gender": ["男", "女", "男", "女", "男", "女", "男", "女"],
    "age": [29, 31, 32, 45, 47, 46, 63, 64],
    "zipcode": ["100", "100", "104", "220", "220", "221", "800", "800"],
    "purchase_amount": [1200, 1500, 1800, 3200, 2800, 3500, 7600, 8100],
    "disease_risk": ["低", "低", "中", "中", "高", "高", "高", "中"]
})

def mask_name(name):
    return name[0] + "○" * (len(name) - 1)

def hash_text(text):
    return hashlib.sha256(text.encode("utf-8")).hexdigest()[:12]

def age_group(age):
    decade = age // 10 * 10
    return f"{decade}-{decade + 9}歲"

def amount_bucket(amount):
    if amount < 2000:
        return "0-1999"
    if amount < 5000:
        return "2000-4999"
    return "5000以上"

anonymous = data.copy()
anonymous["name"] = anonymous["name"].apply(mask_name)
anonymous["email"] = anonymous["email"].apply(hash_text)
anonymous["age"] = anonymous["age"].apply(age_group)
anonymous["purchase_amount_bucket"] = anonymous["purchase_amount"].apply(amount_bucket)
anonymous["purchase_amount_noisy"] = anonymous["purchase_amount"] + np.random.normal(0, 120, len(anonymous)).round(0)
anonymous = anonymous.drop(columns=["purchase_amount"])

print("匿名化後資料：")
display(anonymous)


## 再識別風險與 k-匿名

再識別風險常發生在準識別欄位的組合過於精細時。例如「性別 + 年齡 + 郵遞區號」可能在資料集中只對應到一個人，即使姓名與 Email 已被移除，仍可能被外部資料比對識別。

**k-匿名**的目標是：對每一筆資料而言，在指定的準識別欄位組合下，至少要有 k 筆資料看起來相同。若 k = 2，代表每個人至少能藏在 2 人群組中；若某組只有 1 筆資料，該筆資料就有較高的再識別風險。

本練習用簡化方法計算每筆資料在準識別欄位下的群組大小，藉此判斷是否符合 k-匿名。


In [ ]:
# ── 示範：檢查 k-匿名風險 ────────────────────────────
# 這段程式碼比較精細欄位與泛化欄位對 k-匿名的影響，說明為什麼準識別資訊需要降低精度。

import pandas as pd

raw = pd.DataFrame({
    "gender": ["男", "女", "男", "女", "男", "女", "男", "女"],
    "age": [29, 31, 32, 45, 47, 46, 63, 64],
    "zipcode": ["100", "100", "104", "220", "220", "221", "800", "800"],
    "disease_risk": ["低", "低", "中", "中", "高", "高", "高", "中"]
})

def age_group(age):
    decade = age // 10 * 10
    return f"{decade}-{decade + 9}歲"

def zipcode_generalize(zipcode):
    return zipcode[0] + "**"

def add_k_size(df, quasi_columns):
    result = df.copy()
    result["k_group_size"] = result.groupby(quasi_columns)[quasi_columns[0]].transform("count")
    return result

fine_result = add_k_size(raw, ["gender", "age", "zipcode"])

generalized = raw.copy()
generalized["age"] = generalized["age"].apply(age_group)
generalized["zipcode"] = generalized["zipcode"].apply(zipcode_generalize)
generalized_result = add_k_size(generalized, ["gender", "age", "zipcode"])

print("使用精細準識別欄位時：")
display(fine_result)
print("\n泛化年齡與郵遞區號後：")
display(generalized_result)
print("\n是否符合 k=2：", generalized_result["k_group_size"].min() >= 2)


In [ ]:
# ── 實際應用：合規資料處理檢查表 ──────────────────────────
# 這段程式碼建立一個簡化的 AI 訓練資料合規檢查器，協助判斷資料集是否符合目的限制、資料最小化、保存期限與匿名化要求。

import pandas as pd

checklist = pd.DataFrame({
    "item": [
        "已取得明確同意或具備合法處理依據",
        "資料用途符合原始蒐集目的",
        "未蒐集訓練任務不需要的欄位",
        "已設定保存期限",
        "已移除或匿名化直接識別資訊",
        "已評估準識別欄位的再識別風險",
        "敏感資料有額外保護措施"
    ],
    "status": [True, True, False, True, True, False, True]
})

checklist["result"] = checklist["status"].map({True: "通過", False: "需改善"})
risk_score = (~checklist["status"]).sum()

print("合規檢查結果：")
display(checklist)
print(f"未通過項目數：{risk_score}")

if risk_score == 0:
    print("整體判斷：低風險，可進入後續資料使用流程。")
else:
    failed_items = checklist.loc[~checklist["status"], "item"].tolist()
    print("整體判斷：需改善，請先處理下列項目：")
    for item in failed_items:
        print(f"- {item}")
